In [1]:
"""
MAP - Charting Student Math Misunderstandings - Inference v5.1 (Production Ready)
優化與除錯重點:
1. merge_and_unload 消除 LoRA 運算 overhead
2. Phase 2 改為標準 Right-padding，確保 Token Logits 切片與索引絕對安全
3. Phase 2 引入 SUB_BATCH_SIZE 分批 forward，防止隱藏測試集爆發 OOM
4. 強化防禦性程式碼，避免特殊字串拼接導致 Tokenizer 解析出錯或空列表引發 IndexError
"""

import os
import time
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ==== 路徑 ====
BASE_MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"
ADAPTER_PATH = "/kaggle/input/datasets/alextsai2004/gemma-math-misunderstanding-lora/best_gemma_lora_model"
TEST_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
TRAIN_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
SAMPLE_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/sample_submission.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"

# ==== 超參 ====
BATCH_SIZE = 16           # 同時處理多少筆 test row (Phase 1)
SUB_BATCH_SIZE = 32       # Phase 2 攤平後的 forward 上限，防止 OOM
NUM_BEAMS = 5             # beam search 寬度
NUM_RETURN = 5            # 每筆拿多少候選
MAX_NEW_TOKENS = 24       # label 不會超過這麼長

# ==== Sample submission 骨架 ====
sample = pd.read_csv(SAMPLE_CSV)
ROW_ID_COL = sample.columns[0]
PRED_COL = sample.columns[1]
print(f"Sample shape: {sample.shape}")

# ==== Prompt ====
def build_user_prompt(question, correct_answer, student_explanation):
    return (
        "You are a math misconception classifier.\n"
        "Given the question, the correct answer, and the student's explanation, "
        "predict the final label in the format `Category:Misconception`.\n"
        "If the category is not a misconception type, use `NA` for the misconception part.\n\n"
        f"Question: {question}\n"
        f"Correct answer: {correct_answer}\n"
        f"Student explanation: {student_explanation}\n\n"
        "Return only the label."
    )

def build_prompt(row):
    user = build_user_prompt(
        row["QuestionText"], row["MC_Answer"], row["StudentExplanation"]
    )
    return f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n"

# ==== Model: load + merge LoRA + eval ====
print("Loading base model in bf16 ...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.padding_side = "left"  # Phase 1 生成必須用 left

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"  base loaded in {time.time()-t0:.1f}s")

t0 = time.time()
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print(f"  adapter loaded in {time.time()-t0:.1f}s")

print("Merging LoRA into base ...")
t0 = time.time()
model = model.merge_and_unload()
torch.cuda.empty_cache()  
print(f"  merge done in {time.time()-t0:.1f}s")

model.eval()
model.config.use_cache = True  
device = next(model.parameters()).device

# pad_id 安全 fallback
pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id
    tokenizer.pad_token_id = pad_id

end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
stop_ids = [tokenizer.eos_token_id]
if end_of_turn_id and end_of_turn_id != tokenizer.unk_token_id:
    stop_ids.append(end_of_turn_id)

# ==== 訓練集 labels ====
train_df = pd.read_csv(TRAIN_CSV)
train_df["target"] = (
    train_df["Category"].astype(str) + ":" +
    train_df["Misconception"].fillna("NA").astype(str)
)
unique_labels = sorted(train_df["target"].unique().tolist())
unique_labels_set = set(unique_labels)
fallback_labels = train_df["target"].value_counts().head(3).index.tolist()

# 防禦性確保 fallback 絕對不為空
if not fallback_labels:
    fallback_labels = ["NA:NA", "NA:NA", "NA:NA"]
print(f"# unique labels: {len(unique_labels)}, fallback: {fallback_labels}")

# ==== Test ====
test_df = pd.read_csv(TEST_CSV).reset_index(drop=True)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    test_df[col] = test_df[col].fillna("")
print(f"Test size: {len(test_df)}")
assert len(test_df) == len(sample)

# ==== Helpers ====
def clean_label(text):
    if not text:
        return ""
    label = text.splitlines()[0].strip()
    if " " in label:
        label = label.split(" ")[0]
    return label

@torch.no_grad()
def beam_generate_batch(prompts):
    """批次 beam search (保持 left-padding)"""
    tokenizer.padding_side = "left"
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024,
    ).to(device)
    
    outputs = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        num_return_sequences=NUM_RETURN,
        do_sample=False,
        early_stopping=True,
        eos_token_id=stop_ids,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )
    prompt_len = enc["input_ids"].shape[1]
    outputs = outputs.view(len(prompts), NUM_RETURN, -1)

    results = []
    for i in range(len(prompts)):
        valid, seen = [], set()
        for k in range(NUM_RETURN):
            gen = outputs[i, k, prompt_len:]
            text = tokenizer.decode(gen, skip_special_tokens=True).strip()
            label = clean_label(text)
            if label in unique_labels_set and label not in seen:
                valid.append(label)
                seen.add(label)
        results.append(valid)
    return results

@torch.no_grad()
def score_candidates_batched(prompts, candidates_per_prompt):
    """
    批次計算 log-likelihood。
    修正點: 改用安全穩定的 Right-padding，並加入 SUB_BATCH_SIZE 防禦 OOM。
    """
    flat_sequences = []
    flat_meta = []  # (prompt_idx, cand_idx_in_prompt, label_token_len)

    for pi, (prompt, cands) in enumerate(zip(prompts, candidates_per_prompt)):
        if not cands:
            continue
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=True)
        for ci, cand in enumerate(cands):
            # 安全合併特殊 Token，避免字串拼接被切碎
            cand_ids = tokenizer.encode(cand, add_special_tokens=False)
            if end_of_turn_id:
                cand_ids.append(end_of_turn_id)
            else:
                cand_ids.append(tokenizer.eos_token_id)
                
            flat_sequences.append(prompt_ids + cand_ids)
            flat_meta.append((pi, ci, len(cand_ids)))

    if not flat_sequences:
        return [[] for _ in prompts]

    B = len(flat_sequences)
    max_len = max(len(s) for s in flat_sequences)

    # 建立 Right-padding Tensor
    input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.long)
    label_starts = []
    
    for j, seq in enumerate(flat_sequences):
        input_ids[j, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        attention_mask[j, :len(seq)] = 1
        # 因為是 Right-padding，Label 的起點即為該序列總長度減去 label 長度
        label_starts.append(len(seq) - flat_meta[j][2])

    # 分批 Forward，防止隱藏測試集候選過多時引發 OOM
    all_logits = []
    for m in range(0, B, SUB_BATCH_SIZE):
        sub_input_ids = input_ids[m:m+SUB_BATCH_SIZE].to(device)
        sub_attention_mask = attention_mask[m:m+SUB_BATCH_SIZE].to(device)
        sub_logits = model(input_ids=sub_input_ids, attention_mask=sub_attention_mask).logits
        all_logits.append(sub_logits.cpu()) # 先移回 CPU 緩存節省顯存
        
    logits = torch.cat(all_logits, dim=0)

    # 計算每筆分數
    scores_per_prompt = [[0.0] * len(c) for c in candidates_per_prompt]
    for j, (pi, ci, L) in enumerate(flat_meta):
        ls = label_starts[j]
        slice_logits = logits[j, ls - 1:ls - 1 + L, :].float()
        log_probs = torch.log_softmax(slice_logits, dim=-1)
        
        target_ids = flat_sequences[j][-L:]
        target = torch.tensor(target_ids)
        tok_lp = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
        scores_per_prompt[pi][ci] = tok_lp.mean().item()
        
    return scores_per_prompt

# ==== 主迴圈 ====
print("\nStart inference ...")
pred_dict = {}
phase1_time = 0.0
phase2_time = 0.0
n_empty_after_filter = 0
total_cands = 0

try:
    for start in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Batch"):
        batch_df = test_df.iloc[start:start + BATCH_SIZE]
        prompts = [build_prompt(r) for _, r in batch_df.iterrows()]

        # Phase 1: beam search
        t0 = time.time()
        candidates = beam_generate_batch(prompts)
        phase1_time += time.time() - t0

        # Phase 2: 批次 log-likelihood re-rank
        t0 = time.time()
        scores = score_candidates_batched(prompts, candidates)
        phase2_time += time.time() - t0

        # 取 top-3 + fallback
        for i, (_, row) in enumerate(batch_df.iterrows()):
            cands = candidates[i]
            total_cands += len(cands)
            
            if not cands:
                n_empty_after_filter += 1
                top3 = list(fallback_labels[:3])
            else:
                ranked = sorted(zip(cands, scores[i]), key=lambda x: -x[1])
                top3 = [c for c, _ in ranked]
                for fb in fallback_labels:
                    if len(top3) >= 3:
                        break
                    if fb not in top3:
                        top3.append(fb)
            
            # 防禦性確保絕對補滿 3 個，不使用 top3[0] 避免 IndexError
            while len(top3) < 3:
                top3.append(fallback_labels[0])
                
            pred_dict[row["row_id"]] = " ".join(top3[:3])

except Exception as e:
    print(f"\n[CRITICAL ERROR] Inference broken: {str(e)}")
    print("Generating safe dummy submission to avoid absolute failure...")
    # 萬一不幸卡進不可抗力的錯誤，輸出一個合法格式的 fallback 答案，至少拿到基本分數
    for _, row in test_df.iterrows():
        if row["row_id"] not in pred_dict:
            pred_dict[row["row_id"]] = " ".join(fallback_labels[:3])

# ==== Diagnostic ====
print(f"\nTiming:")
print(f"  Phase 1 (beam):    {phase1_time:.1f}s")
print(f"  Phase 2 (re-rank): {phase2_time:.1f}s")
print(f"  Avg candidates per row: {total_cands/len(test_df):.2f}")
print(f"  Rows with 0 valid candidates: {n_empty_after_filter} "
      f"({n_empty_after_filter/len(test_df)*100:.1f}%)")

# ==== 用 sample 骨架建 submission ====
submission = sample.copy()
submission[PRED_COL] = submission[ROW_ID_COL].map(pred_dict)

# 萬一有漏掉的 row_id，全面強制填補
if submission[PRED_COL].isna().any():
    submission[PRED_COL] = submission[PRED_COL].fillna(" ".join(fallback_labels[:3]))

print("\nValidation:")
print(f"Shape: {submission.shape}")
print(f"Any NaN: {submission.isna().any().any()}")
print(f"Head:\n{submission.head()}")

assert submission.shape == sample.shape
assert not submission.isna().any().any()
assert (submission[PRED_COL] != "").all()
assert (submission[PRED_COL].str.split().str.len() == 3).all()

submission.to_csv(OUTPUT_CSV, index=False)
print(f"\n[OK] Saved {OUTPUT_CSV}")

Sample shape: (3, 2)
Loading base model in bf16 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

  base loaded in 26.2s


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


  adapter loaded in 2.4s
Merging LoRA into base ...
  merge done in 0.2s
# unique labels: 65, fallback: ['True_Correct:NA', 'False_Neither:NA', 'True_Neither:NA']
Test size: 3

Start inference ...


Batch: 100%|██████████| 1/1 [00:05<00:00,  5.74s/it]


Timing:
  Phase 1 (beam):    2.9s
  Phase 2 (re-rank): 2.9s
  Avg candidates per row: 4.00
  Rows with 0 valid candidates: 0 (0.0%)

Validation:
Shape: (3, 2)
Any NaN: False
Head:
   row_id                             Category:Misconception
0   36696  True_Neither:NA True_Correct:NA True_Misconcep...
1   36697  False_Neither:NA False_Misconception:WNB False...
2   36698   True_Neither:NA True_Correct:NA False_Neither:NA

[OK] Saved /kaggle/working/submission.csv
